# MemSysExplorer Profiler Demo

This notebook demonstrates how to use the MemSysExplorer profiling system with different profilers.

**Profilers covered:**
1. DynamoRIO - Dynamic Binary Instrumentation (CPU)
2. Sniper - Architectural Simulator (CPU)
3. Perf - Hardware Performance Counters (CPU)
4. NVBit - NVIDIA Dynamic Binary Instrumentation (GPU)
5. Nsight Compute - NVIDIA Hardware Performance Counters (GPU)

---
## 1. Main Interface Help

The `main.py` script is the entry point for all profilers.

In [ ]:
!python3 main.py --help

---
## 2. Sample Test Programs

We use simple test programs for CPU and GPU profiling.

### 2.1 CPU Sample Program

A simple test program that performs a loop with sqrt calculations.

In [ ]:
!cat test/sample.c

### 2.2 GPU Sample Program

A simple CUDA program that performs vector addition on the GPU.

In [ ]:
!cat test/sample_gpu.cu

### 2.3 Build Test Programs

In [ ]:
# Build the CPU sample program
!cd test && make cpu

In [ ]:
# Build the GPU sample program (requires CUDA toolkit)
!cd test && make gpu

---
---
# CPU Profilers

---
# 3. DynamoRIO Profiler

DynamoRIO is a Dynamic Binary Instrumentation (DBI) tool that tracks memory access patterns without modifying source code.

## 3.1 DynamoRIO Configuration

The memcount client uses a text-based configuration file.

In [ ]:
!cat config/memcount_config.txt

## 3.2 Run DynamoRIO Profiler

**Common Command:**
```bash
python3 main.py --profiler dynamorio --action both --executable <your_executable>
```

In [ ]:
# Clean up previous outputs before DynamoRIO run
!rm -f memsyspatternconfig_sample*.json memsysmetadata_dynamorio*.json

In [ ]:
# Run DynamoRIO profiler with the sample program
!python3 main.py --profiler dynamorio --action both --config config/memcount_config.txt --executable test/sample

## 3.3 DynamoRIO Output

View the generated PatternConfig output.

In [ ]:
import json
import glob
import os

# Find DynamoRIO pattern config (named after executable: sample)
drio_files = sorted(glob.glob("memsyspatternconfig_sample*.json"), key=os.path.getmtime, reverse=True)
if drio_files:
    with open(drio_files[0]) as f:
        data = json.load(f)
    print(f"File: {drio_files[0]}")
    print(json.dumps(data, indent=2))
else:
    print("No pattern config found. Run the profiler first.")

## 3.4 DynamoRIO Metadata

View the system metadata captured during profiling.

In [ ]:
import json
import glob
import os

# Find DynamoRIO metadata
meta_files = sorted(glob.glob("memsysmetadata_dynamorio*.json"), key=os.path.getmtime, reverse=True)
if meta_files:
    with open(meta_files[0]) as f:
        data = json.load(f)
    print(f"File: {meta_files[0]}")
    print(json.dumps(data, indent=2))

---
---
# 4. Sniper Profiler

Sniper is a cycle-accurate architectural simulator that models multi-core systems with detailed memory hierarchy.

## 4.1 Sniper Configuration

Sniper uses `.cfg` files to define CPU and memory hierarchy parameters. The `gainestown` config models an Intel Xeon X5550.

In [ ]:
!cat config/gainestown.cfg

## 4.2 Run Sniper Profiler

**Common Command:**
```bash
python3 main.py -p sniper -a both --config gainestown --level l3 --results_dir . --executable <your_executable>
```

In [ ]:
# Clean up previous outputs before Sniper run
!rm -f memsyspatternconfig_sample*.json memsysmetadata_sniper*.json

In [ ]:
# Run Sniper profiler with the sample program
!python3 main.py -p sniper -a both --config gainestown --level l3 --results_dir . --executable test/sample

## 4.3 Sniper Output

View the generated PatternConfig output. Sniper generates per-core statistics.

In [ ]:
import json
import glob
import os

# Find Sniper pattern config (named after executable: sample)
# Sniper runs after DynamoRIO, so look for most recent sample*.json
sniper_files = sorted(glob.glob("memsyspatternconfig_sample*.json"), key=os.path.getmtime, reverse=True)
if sniper_files:
    with open(sniper_files[0]) as f:
        data = json.load(f)
    print(f"File: {sniper_files[0]}")
    print(json.dumps(data, indent=2))

## 4.4 Sniper Metadata

View the system metadata captured during simulation.

In [ ]:
import json
import glob
import os

# Find Sniper metadata
meta_files = sorted(glob.glob("memsysmetadata_sniper*.json"), key=os.path.getmtime, reverse=True)
if meta_files:
    with open(meta_files[0]) as f:
        data = json.load(f)
    print(f"File: {meta_files[0]}")
    print(json.dumps(data, indent=2))

---
---
# 5. Perf Profiler

Perf leverages **Hardware Performance Counters (HPC)** to collect low-level hardware metrics from CPUs. It provides insights into cache behavior at different memory hierarchy levels.

**Note:** Perf requires Linux with access to hardware performance counters. It may not work in WSL or virtualized environments.

## 5.1 Perf Profiler Help

View the available options for the Perf profiler.

In [ ]:
!python3 main.py --profiler perf --action both --help

## 5.2 Run Perf Profiler

**Common Command:**
```bash
python3 main.py -p perf -a both --level l2 --executable <your_executable>
```

**Note:** May require `sudo sh -c 'echo -1 > /proc/sys/kernel/perf_event_paranoid'` for HPC access.

In [ ]:
# Clean up previous outputs before Perf run
!rm -f memsyspatternconfig_sample*.json memsysmetadata_perf*.json

In [ ]:
# Run Perf profiler with the sample program (L2 cache level)
!python3 main.py -p perf -a both --level l2 --executable test/sample

## 5.3 Perf Output

View the generated PatternConfig output.

In [ ]:
import json
import glob
import os

# Find Perf pattern config (named after executable: sample)
perf_files = sorted(glob.glob("memsyspatternconfig_sample*.json"), key=os.path.getmtime, reverse=True)
if perf_files:
    with open(perf_files[0]) as f:
        data = json.load(f)
    print(f"File: {perf_files[0]}")
    print(json.dumps(data, indent=2))

## 5.4 Perf Metadata

View the system metadata captured during profiling.

In [ ]:
import json
import glob
import os

# Find Perf metadata
meta_files = sorted(glob.glob("memsysmetadata_perf*.json"), key=os.path.getmtime, reverse=True)
if meta_files:
    with open(meta_files[0]) as f:
        data = json.load(f)
    print(f"File: {meta_files[0]}")
    print(json.dumps(data, indent=2))

---
---
# GPU Profilers

The following profilers are designed for NVIDIA GPU workloads. They require:
- NVIDIA GPU with CUDA support
- CUDA Toolkit installed
- For NVBit: the `nvbit.so` library built
- For Nsight Compute: `ncu` command available in PATH

---
# 6. NVBit Profiler

NVBit is NVIDIA's binary instrumentation framework that allows dynamic instrumentation of CUDA applications. It tracks memory access patterns at the GPU level by injecting instrumentation code via `LD_PRELOAD`.

**Requirements:**
- Built `nvbit.so` library at `profilers/nvbit/lib/nvbit.so`
- CUDA-enabled GPU
- CUDA executable to profile

## 6.1 Run NVBit Profiler

**Common Command:**
```bash
python3 main.py --profiler nvbit --action both --executable <your_cuda_executable>
```

NVBit uses `LD_PRELOAD` to inject the instrumentation library at runtime. The profiler produces a `global_summary.txt` file with memory access statistics.

In [ ]:
# Clean up previous outputs before NVBit run
!rm -f memsyspatternconfig_sample_gpu*.json memsysmetadata_nvbit*.json global_summary.txt

In [ ]:
# Run NVBit profiler with the GPU sample program
!python3 main.py --profiler nvbit --action both --executable test/sample_gpu

## 6.2 NVBit Raw Output

View the raw `global_summary.txt` output generated by NVBit instrumentation.

In [ ]:
!cat global_summary.txt 2>/dev/null || echo "global_summary.txt not found. Run the NVBit profiler first."

## 6.3 NVBit PatternConfig Output

View the generated PatternConfig JSON output.

In [ ]:
import json
import glob
import os

# Find NVBit pattern config
nvbit_files = sorted(glob.glob("memsyspatternconfig_sample_gpu*.json"), key=os.path.getmtime, reverse=True)
if nvbit_files:
    with open(nvbit_files[0]) as f:
        data = json.load(f)
    print(f"File: {nvbit_files[0]}")
    print(json.dumps(data, indent=2))
else:
    print("No NVBit pattern config found. Run the profiler first.")

## 6.4 NVBit Metadata

View the system metadata captured during GPU profiling.

In [ ]:
import json
import glob
import os

# Find NVBit metadata
meta_files = sorted(glob.glob("memsysmetadata_nvbit*.json"), key=os.path.getmtime, reverse=True)
if meta_files:
    with open(meta_files[0]) as f:
        data = json.load(f)
    print(f"File: {meta_files[0]}")
    print(json.dumps(data, indent=2))
else:
    print("No NVBit metadata found. Run the profiler first.")

---
---
# 7. Nsight Compute (NCU) Profiler

Nsight Compute is NVIDIA's profiling tool that uses **Hardware Performance Counters** to collect detailed GPU metrics. It provides insights into kernel execution, memory bandwidth, cache hit rates, and more.

**Requirements:**
- NVIDIA Nsight Compute installed (`ncu` in PATH)
- CUDA-enabled GPU
- Configuration file specifying metrics to collect

## 7.1 NCU Configuration

NCU uses `.conf` files to specify which metrics to collect. The profiler supports three levels:
- `l2` - L2 cache metrics
- `dram` - DRAM bandwidth metrics
- `custom` - User-defined metrics

Config files are stored in `profilers/ncu/configs/`. Below is the custom configuration:

In [ ]:
!cat profilers/ncu/configs/custom_settings.conf

### Available Metric Levels

You can also view the L2 and DRAM-specific configurations:

In [ ]:
print("=== L2 Settings ===")
!cat profilers/ncu/configs/l2_settings.conf
print("\n=== DRAM Settings ===")
!cat profilers/ncu/configs/dram_settings.conf

## 7.2 Run Nsight Compute Profiler

**Common Command:**
```bash
python3 main.py -p ncu -a both --level custom --executable <your_cuda_executable>
```

**Level options:**
- `--level l2` - Collect L2 cache metrics
- `--level dram` - Collect DRAM bandwidth metrics
- `--level custom` - Collect user-defined metrics from `custom_settings.conf`

**Note:** NCU generates per-kernel statistics. Each CUDA kernel invocation produces separate metrics.

In [ ]:
# Clean up previous outputs before NCU run
!rm -f memsyspatternconfig_*.json memsysmetadata_ncu*.json *.ncu-rep

In [ ]:
# Run Nsight Compute profiler with the GPU sample program (custom level)
!python3 main.py -p ncu -a both --level custom --executable test/sample_gpu

## 7.3 NCU PatternConfig Output

View the generated PatternConfig JSON output. NCU generates per-kernel results, so you may see multiple output files (one per kernel).

In [ ]:
import json
import glob
import os

# Find NCU pattern configs (may have multiple for different kernels)
ncu_files = sorted(glob.glob("memsyspatternconfig_*.json"), key=os.path.getmtime, reverse=True)
for ncu_file in ncu_files:
    with open(ncu_file) as f:
        data = json.load(f)
    print(f"\n=== File: {ncu_file} ===")
    print(json.dumps(data, indent=2))

## 7.4 NCU Metadata

View the system metadata captured during GPU profiling.

In [ ]:
import json
import glob
import os

# Find NCU metadata
meta_files = sorted(glob.glob("memsysmetadata_ncu*.json"), key=os.path.getmtime, reverse=True)
if meta_files:
    with open(meta_files[0]) as f:
        data = json.load(f)
    print(f"File: {meta_files[0]}")
    print(json.dumps(data, indent=2))
else:
    print("No NCU metadata found. Run the profiler first.")

## 7.5 NCU Report File

NCU also generates a `.ncu-rep` file that can be opened in the Nsight Compute GUI for detailed analysis.

In [ ]:
!ls -la *.ncu-rep 2>/dev/null || echo "No .ncu-rep files found."

---
---
# 8. Summary

This notebook demonstrated how to use MemSysExplorer with five different profilers:

| Profiler | Type | Platform | Key Features |
|----------|------|----------|-------------|
| DynamoRIO | DBI | CPU | Memory access tracing, working set size |
| Sniper | Simulator | CPU | Cycle-accurate, cache hierarchy modeling |
| Perf | HPC | CPU | Hardware counters, cache statistics |
| NVBit | DBI | GPU | GPU memory instrumentation, LD_PRELOAD |
| Nsight Compute | HPC | GPU | Per-kernel metrics, L2/DRAM analysis |

All profilers produce standardized PatternConfig JSON files that can be used for memory system analysis.